In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import geopandas as gpd
import seaborn as sns
from libpysal.weights import DistanceBand, KNN
from esda.moran import Moran
from spreg import OLS, ML_Error, ML_Lag, GM_Error, GM_Lag
from scipy import stats
from shapely.geometry import Point
from splot.esda import moran_scatterplot, plot_moran
import contextily as ctx
from matplotlib.ticker import FuncFormatter
import pyproj
from libpysal.weights import lag_spatial
from scipy.stats import norm
from statsmodels.stats.outliers_influence import variance_inflation_factor

import libpysal as ps 
from libpysal.weights import Queen
from esda.moran import Moran
import statsmodels.api as sm
# MGWR functions
from mgwr.gwr import GWR,MGWR
from mgwr.sel_bw import Sel_BW

In [ ]:
# Read the dataset and check the structure of the data:
df = pd.read_csv(r'Edinburgh_ABnB_List_Data\listings.csv')
df.info()

In [ ]:
# Select columns relevant to the regression analysis and check for missing values:
columns_to_keep = ['latitude', 'longitude', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'price',
                   'number_of_reviews', 'review_scores_rating', 'host_identity_verified','host_is_superhost']

df = df[columns_to_keep].copy()

df.head()

for col in columns_to_keep:
    print(f"{col}: {df[col].isnull().sum()} null values")

In [ ]:
# Clean price column and convert to float:
df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)
# Convert binary variables, t or f, (host_identity_verified and host_is_superhost) to numeric (0, 1):
df['host_identity_verified_binary'] = df['host_identity_verified'].map({'t': 1, 'f': 0})
df['host_is_superhost_binary'] = df['host_is_superhost'].map({'t': 1, 'f': 0})

# Separate room_type into dummy variables:
df['entire_home_apt'] = (df['room_type'] == 'Entire home/apt').astype(int)
df['private_room'] = (df['room_type'] == 'Private room').astype(int)
df['shared_room'] = (df['room_type'] == 'Shared room').astype(int)

df = df.drop(columns=['room_type', 'host_identity_verified', 'host_is_superhost'])

# Remove NA values and review the updated dataset:
updated_cols = df.columns
df_updated = df.dropna(subset = updated_cols)

df_updated.describe()

In [ ]:
variables = ['price', 'bathrooms', 'bedrooms', 'accommodates']
titles = ['Price (£)', 'Bathrooms (No.)', 'Bedrooms (No.)', 'Accommodates (No.)']


fig, axes = plt.subplots(2, 2, figsize=(20, 16))
axes = axes.flatten()

for i, (col, title) in enumerate(zip(variables, titles)):
    ax = axes[i]
    df_updated.boxplot(column=[col], ax=ax, grid=False)
    ax.set_title(f'Distribution: {title}', fontsize=14)
    ax.set_ylabel(f'{title}')

plt.suptitle('Box Plot Distribution of Variables', fontsize=18, fontweight='bold', y=1.01)
plt.show()

In [ ]:
# Remove outliers from the price column using the 95th percentile:
price_95th_percentile = df_updated['price'].quantile(0.95)
print(f"Number of price records removed: {len(df_updated[df_updated['price'] > price_95th_percentile])}")
df_updated = df_updated[df_updated['price'] <= price_95th_percentile]

# Remove outliers from accommodates, bathrooms, bedrooms:
print(f"Number of bedrooms records removed: {len(df_updated[df_updated['bedrooms'] == 0])}")
df_updated = df_updated[df_updated['bedrooms'] > 0]

print(f"Number of bathrooms records removed: {len(df_updated[df_updated['bathrooms'] == 0])}")
df_updated = df_updated[df_updated['bathrooms'] > 0]

print(f"final number of observations for modeling = {len(df_updated)}")

In [ ]:
# Isolate the key independent variables for the regression model:
x_cols = ['accommodates', 'bathrooms', 'bedrooms', 'number_of_reviews',
          'review_scores_rating', 'host_identity_verified_binary', 'host_is_superhost_binary',
            'entire_home_apt', 'private_room', 'shared_room']

# Create a new DataFrame with the independent variables for VIF calculation:
X_vif = df_updated[x_cols] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data.sort_values('VIF', ascending=False))

In [ ]:
x_cols = ['bedrooms', 'number_of_reviews', 'host_is_superhost_binary',
            'entire_home_apt', 'private_room', 'shared_room']

X_vif = df_updated[x_cols] 

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print(vif_data.sort_values('VIF', ascending=False))

In [ ]:
geometry = [Point(xy) for xy in zip(df_updated['longitude'], df_updated['latitude'])]
# initial data are in WGS84
gdf = gpd.GeoDataFrame(df_updated, geometry=geometry, crs='EPSG:4326')

# convert listings to web Mercator for plotting with basemap
gdf = gdf.to_crs(epsg=3857)

# load neighbourhood boundaries and transform as well:
neigh = gpd.read_file(r'Edinburgh_ABnB_List_Data\neighbourhoods.geojson')
neigh_web = neigh.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(10, 8))

# plot neighbourhood boundaries:
neigh_web.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5, alpha=0.5)

# Plot the listings colours by price using web Mercator geometry
x = gdf.plot(
    ax=ax, column='price', cmap='YlOrRd',
    markersize=5, alpha=0.6, legend=True,
    legend_kwds={'label': 'Price (£)', 'shrink': 0.7}
)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_xlabel('Lon')
ax.set_ylabel('Lat')

transformer = pyproj.Transformer.from_crs('EPSG:3857', 'EPSG:4326', always_xy=True)

def lon_formatter(x, pos):
    lon, _ = transformer.transform(x, 0)
    return f'{lon:.2f}°'

def lat_formatter(y, pos):
    _, lat = transformer.transform(0, y)
    return f'{lat:.2f}°'

ax.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
ax.yaxis.set_major_formatter(FuncFormatter(lat_formatter))

plt.tight_layout()
plt.show()

In [ ]:
# Convert to appropriate coordinates for distance calculations:
gdf_proj = gdf.to_crs('EPSG:27700')
coords = np.column_stack((gdf_proj.geometry.x, gdf_proj.geometry.y))

# ---- KNN Weight Matrix ----
w_knn = KNN.from_array(coords, k=12)
w_knn.transform = 'R'

print("=== KNN Spatial Weights (k=12) ===")
print(f"Number of observations: {w_knn.n}")
print(f"Mean number of neighbours: {w_knn.mean_neighbors:.1f}")
print(f"Min neighbours: {w_knn.min_neighbors}")
print(f"Max neighbours: {w_knn.max_neighbors}")

In [ ]:
sem = ML_Error(
    ys, xs,
    w=w_knn,
    name_y=y_col,
    name_x=x_cols,
    name_ds='Edinburgh Airbnb Listings'
)

print(sem.summary)

# Moran's I test for spatial autocorrelation in SEM residuals
residuals = sem.u.flatten()
moran = Moran(residuals, w_knn)

print("=== Global Moran's I Test on SEM Residuals ===")
print(f"Moran's I:     {moran.I:.4f}")
print(f"Expected I:    {moran.EI:.4f}")
print(f"Z-score:       {moran.z_sim:.4f}")
print(f"P-value:       {moran.p_sim:.4f}")
print()